In [17]:
import pandas as pd
from sqlalchemy import create_engine,URL,text

In [18]:
def data_extraction():
    try:
        connection_url = URL.create(
            drivername="mysql+pymysql",
            username="root",
            password="Shantanu@2026",
            host="localhost",
            port=3306,
            database="test_case_2"
        )

        engine = create_engine(connection_url)

        customer = pd.read_sql("SELECT * FROM customers", engine)
        orders = pd.read_sql("SELECT * FROM orders", engine)
        order_item = pd.read_sql("SELECT * FROM order_items", engine)
        product = pd.read_sql("SELECT * FROM products", engine)

        return customer, orders, order_item, product

    except Exception as e:
        print(f"Error: {e}")

In [19]:

def transformation(customer, order, order_item, product):

    # Remove duplicates
    customer.drop_duplicates(inplace=True)
    order.drop_duplicates(inplace=True)
    order_item.drop_duplicates(inplace=True)
    product.drop_duplicates(inplace=True)

    # Customer transformation
    customer = customer[customer["customer_name"].notna()]
    customer["city"] = customer["city"].fillna("Unknown")
    customer = customer[customer["email"].notna()]

    # Order transformation
    order = order[order["customer_id"].isin(customer["customer_id"])]
    order["order_date"] = order["order_date"].ffill()

    # Order item transformation
    order_item = order_item[order_item["order_id"].notna()]
    order_item = order_item[order_item["product_id"].notna()]

    order_item = order_item[
        order_item["order_id"].isin(order["order_id"])
    ]

    order_item.loc[order_item["quantity"] <= 0, "quantity"] = 1

    order_item = order_item[
        order_item["product_id"].isin(product["product_id"])
    ]

    # Product transformation
    product = product[product["product_id"].notna()]
    product["product_name"] = product["product_name"].fillna("Unknown")
    product["category"] = product["category"].fillna("Unknown")
    product = product[product["price"] > 0]

    return customer, order, order_item, product



In [20]:
def loading_data(customer, order, order_item, product):
        try:
            connection_url_2 = URL.create(
                drivername="mysql+pymysql",
                username="root",
                password="Shantanu@2026",
                host="localhost",
                port=3306,
                database="test_case_2_dw"
            )

            engine2 = create_engine(connection_url_2)


            with engine2.begin() as conn:

                  # 1. Customers
                  customer.to_sql(
                  name="stagging_customers",
                  con=conn,
                  if_exists="append",
                  index=False
                  )

                  # 2. Products
                  product.to_sql(
                  name="stagging_products",
                  con=conn,
                  if_exists="append",
                  index=False
                  )

                  # 3. Orders
                  order.to_sql(
                  name="stagging_orders",
                  con=conn,
                  if_exists="append",
                  index=False
                  )

                  # 4. Order Items
                  order_item.to_sql(
                  name="stagging_order_items",
                  con=conn,
                  if_exists="append",
                  index=False
                  )

                  # Load dimension tables
                  conn.execute(text("""
                        INSERT INTO customer_dim
                              (customer_id, customer_name, city, email)
                        SELECT
                              customer_id,
                              customer_name,
                              city,
                              email
                        FROM stagging_customers
                  """))

                  conn.execute(text("""
                        INSERT INTO order_dim
                              (order_id, order_date)
                        SELECT
                              order_id,
                              order_date
                        FROM stagging_orders
                  """))

                  conn.execute(text("""
                  INSERT INTO product_dim
                        (product_id, product_name, category)
                  SELECT
                        product_id,
                        product_name,
                        category
                  FROM stagging_products
                  """))

                  conn.execute(text(
                        """INSERT INTO fact_total_sale (order_key,product_key,customer_key,price,quantity,total_sale)
                        SELECT o.order_key,p.product_key,c.customer_key,pr.price,soi.quantity,pr.price*soi.quantity as total_sale
                        FROM stagging_order_items soi
                        JOIN stagging_orders so
                              on so.order_id = soi.order_id 
                        JOIN order_dim o
                              on o.order_id=so.order_id
                        JOIN customer_dim c
                              on so.customer_id = c.customer_id
                        JOIN  product_dim p
                              on soi.product_id = p.product_id
                        JOIN stagging_products pr
                        on soi.product_id =pr.product_id;
                        """
                  ))

                  

        except Exception as e:
              print(f"Error : {e}")

    

In [21]:
def main_function():
    try:
        customer, order, order_item, product=data_extraction()
        customer, order, order_item, product= transformation(customer, order, order_item, product)
    except Exception as e:
        print(f"Error : {e}")

    

In [22]:
main_function()

Error: (pymysql.err.OperationalError) (1045, "Access denied for user 'root'@'localhost' (using password: YES)")
(Background on this error at: https://sqlalche.me/e/21/e3q8)
Error : cannot unpack non-iterable NoneType object


In [26]:
from getpass import getpass
from sqlalchemy import create_engine, URL, text

password = getpass("Enter ETL password:")

engine = create_engine(
    URL.create(
        "mysql+pymysql",
        username="etl_user",
        password=password,
        host="localhost",
        port=3306,
        database="test_case_2"
    )
)

with engine.connect() as conn:
    print(conn.execute(
        text("SELECT DATABASE(), USER()")
    ).fetchone())

engine.dispose()

OperationalError: (pymysql.err.OperationalError) (1045, "Access denied for user 'etl_user'@'localhost' (using password: YES)")
(Background on this error at: https://sqlalche.me/e/21/e3q8)

In [ ]:
password = getpass("Enter ETL password: ")

In [ ]:
from getpass import getpass
from sqlalchemy import create_engine, URL, text

password = getpass("Enter MySQL Workbench ROOT password:aspirinee00079")

engine = create_engine(
    URL.create(
        drivername="mysql+pymysql",
        username="root",
        password=password,
        host="localhost",
        port=3306,
        database="test_case_2"
    ),
    connect_args={"connect_timeout": 5}
)

with engine.connect() as conn:
    print("MYSQL CONNECTION SUCCESSFUL!")
    print(conn.execute(
        text("SELECT VERSION(), DATABASE(), USER()")
    ).fetchone())

engine.dispose()

MYSQL CONNECTION SUCCESSFUL!
('8.4.11', 'test_case_2', 'root@localhost')


In [24]:
password = getpass("Enter MySQL ROOT password: ")

In [27]:
from sqlalchemy import create_engine, URL, text

engine = create_engine(
    URL.create(
        drivername="mysql+pymysql",
        username="root",
        password=password,
        host="localhost",
        port=3306,
        database="test_case_2"
    ),
    pool_pre_ping=True,
    connect_args={"connect_timeout": 5}
)

with engine.connect() as conn:
    result = conn.execute(
        text("SELECT VERSION(), DATABASE(), USER()")
    )
    print("MYSQL CONNECTION SUCCESSFUL!")
    print(result.fetchone())

engine.dispose()

OperationalError: (pymysql.err.OperationalError) (1045, "Access denied for user 'root'@'localhost' (using password: YES)")
(Background on this error at: https://sqlalche.me/e/21/e3q8)

In [32]:
from getpass import getpass
from sqlalchemy import create_engine, URL, text

password = getpass("Enter ETL User Password: ")

engine = create_engine(
    URL.create(
        drivername="mysql+pymysql",
        username="etl_user",
        password=password,
        host="127.0.0.1",
        port=3306,
        database="test_case_2"
    ),
    pool_pre_ping=True,
    connect_args={"connect_timeout": 5}
)

try:
    with engine.connect() as conn:
        print("MYSQL CONNECTION SUCCESSFUL!")
        result = conn.execute(
            text("SELECT VERSION(), DATABASE(), USER()")
        )
        print(result.fetchone())

except Exception as e:
    print("CONNECTION FAILED:", e)

finally:
    engine.dispose()

CONNECTION FAILED: (pymysql.err.OperationalError) (1045, "Access denied for user 'etl_user'@'localhost' (using password: YES)")
(Background on this error at: https://sqlalche.me/e/21/e3q8)


In [33]:
from sqlalchemy import create_engine, URL, text

engine = create_engine(
    URL.create(
        "mysql+pymysql",
        username="etl_user",
        password="OmCase2@2026!",
        host="127.0.0.1",
        port=3306,
        database="test_case_2"
    )
)

with engine.connect() as conn:
    print("MYSQL CONNECTION SUCCESSFUL!")
    print(conn.execute(
        text("SELECT VERSION(), DATABASE(), USER()")
    ).fetchone())

engine.dispose()

MYSQL CONNECTION SUCCESSFUL!
('8.4.11', 'test_case_2', 'etl_user@localhost')


In [36]:
password = getpass("Enter ETL password: ")

connection_url = URL.create(
    drivername="mysql+pymysql",
    username="etl_user",
    password=password,
    host="127.0.0.1",
    port=3306,
    database="test_case_2"
)

engine = create_engine(
    connection_url,
    pool_pre_ping=True
)

from sqlalchemy import text

try:
    with engine.connect() as conn:
        result = conn.execute(
            text("SELECT VERSION(), DATABASE(), USER()")
        )
        print("CONNECTION SUCCESSFUL!")
        print(result.fetchone())

except Exception as e:
    print("CONNECTION FAILED:", e)

CONNECTION FAILED: (pymysql.err.OperationalError) (1045, "Access denied for user 'etl_user'@'localhost' (using password: YES)")
(Background on this error at: https://sqlalche.me/e/21/e3q8)


In [35]:
from sqlalchemy import text

try:
    with engine.connect() as conn:
        result = conn.execute(
            text("SELECT VERSION(), DATABASE(), USER()")
        )
        print("CONNECTION SUCCESSFUL!")
        print(result.fetchone())

except Exception as e:
    print("CONNECTION FAILED:", e)

CONNECTION FAILED: (pymysql.err.OperationalError) (1045, "Access denied for user 'etl_user'@'localhost' (using password: YES)")
(Background on this error at: https://sqlalche.me/e/21/e3q8)


In [37]:
password = getpass("Enter ETL password: ")

In [ ]:
from sqlalchemy import create_engine, URL, text

engine = create_engine(
    URL.create(
        drivername="mysql+pymysql",
        username="etl_user",
        password=password,
        host="127.0.0.1",
        port=3306,
        database="test_case_2"
    ),
    pool_pre_ping=True
)

try:
    with engine.connect() as conn:
        print("MYSQL CONNECTION SUCCESSFUL!")
        print(
            conn.execute(
                text("SELECT VERSION(), DATABASE(), USER()")
            ).fetchone()
        )
except Exception as e:
    print("CONNECTION FAILED:", e)
finally:
    engine.dispose()

CONNECTION FAILED: (pymysql.err.OperationalError) (1045, "Access denied for user 'etl_user'@'localhost' (using password: YES)")
(Background on this error at: https://sqlalche.me/e/21/e3q8)
